In [4]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import itertools
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL

from ccm_core import (cross_map_rho, ebisuzaki_surrogates, select_E,
                      delay_embed, simplex_predict)

warnings.filterwarnings("ignore")

# Configuration
ROOT = os.environ.get("AGL_ROOT", ".")
LEVELS = os.path.join(ROOT, "Code Outputs", "Gap Interpolation Outputs",
                      "Unified_Interpolated_Levels.xlsx")
EDA_CORR = os.path.join(ROOT, "Code Outputs", "EDA Outputs",
                        "EDA_06_corr_residual.csv")
EDA_LEAD = os.path.join(ROOT, "Code Outputs", "EDA Outputs",
                        "EDA_09_leadlag_summary.csv")
OUTDIR = os.path.join(ROOT, "Code Outputs", "CCM Outputs")
os.makedirs(OUTDIR, exist_ok=True)

WIN_START, WIN_END = "1995-06-01", "2025-12-01"
TRAIN_END = "2020-12-01"

E_RANGE = range(1, 11)
E_TOL = 0.01            # parsimony tolerance for E (see ccm_core.select_E)
TOP_K = 3               # neighbours per node in the sparse GNAR-ready adjacency
TAU = 1
N_BOOT = 100            # library replicates per library size
N_SURR = 200            # Ebisuzaki surrogates for the significance test
EXCL = 0                # Theiler radius for the main run (0 = exclude self only)
EXCL_SENS = 12          # sensitivity run: exclude +/- 12 months
SEED = 20260807
ALPHA = 0.05
RHO_FLOOR = 0.10        # minimum terminal rho for an edge to be called
N_CONTROL = 200         # null trials for the false-positive-rate estimate
PARALLEL = False        

SHORT = {"Lake Albert": "Alb", "Lake Edward": "Edw", "Lake Kivu": "Kiv",
         "Lake Malawi": "Mal", "Lake Tanganyika": "Tan",
         "Lake Turkana": "Tur", "Lake Victoria": "Vic"}


# data
def load_levels():
    df = pd.read_excel(LEVELS)
    W = df.pivot(index="Date", columns="Reservoir", values="Level_m")
    W.index = pd.to_datetime(W.index)
    W = W.loc[WIN_START:WIN_END].sort_index()
    if W.isna().any().any():
        raise SystemExit("NaNs inside the canonical window.")
    if len(W) != 367:
        raise SystemExit(f"expected 367 months, got {len(W)}")
    return W


def make_variant(W, preproc, window):
    X = W.loc[:TRAIN_END] if window == "train" else W
    out = {}
    for lake in X.columns:
        s = X[lake].astype(float)
        if preproc == "stl":
            r = STL(s, period=12, robust=True).fit().resid
            out[lake] = np.asarray(r, dtype=float)
        elif preproc == "diff":
            out[lake] = np.asarray(s.diff().dropna(), dtype=float)
        else:
            raise ValueError(preproc)
    idx = X.index if preproc == "stl" else X.index[1:]
    D = pd.DataFrame(out, index=idx)
    # standardise so rho is not affected by units
    return (D - D.mean()) / D.std(ddof=0)


# run one variant
def _map_jobs(fn, jobs):

    if PARALLEL:
        try:
            from multiprocessing import Pool, cpu_count
            with Pool(min(cpu_count(), 8)) as pool:
                return pool.map(fn, jobs)
        except Exception:
            pass
    return [fn(j) for j in jobs]


def run_variant(D, preproc, window, log, delta_thresh):
    lakes = list(D.columns)
    n = len(D)

    # embedding dimension per lake (simplex self-prediction, tp=1)
    Es, Ecurves = {}, []
    for lk in lakes:
        E, r, curve = select_E(D[lk].values, E_RANGE, TAU, EXCL, tol=E_TOL)
        finite = {k: v for k, v in curve.items() if np.isfinite(v)}
        E_arg = max(finite, key=finite.get)
        Es[lk] = E
        for e, rr in curve.items():
            Ecurves.append({"preproc": preproc, "window": window, "Lake": lk,
                            "E": e, "simplex_rho": rr, "selected": e == E,
                            "argmax": e == E_arg})
        log(f"    E({SHORT[lk]})={E} rho={r:.3f}"
            f"   [argmax would be E={E_arg}, rho={finite[E_arg]:.3f}]")

    # library sizes
    Emax = max(Es.values())
    Lmax = n - (Emax - 1) * TAU
    Ls = sorted({L for L in [20, 30, 45, 65, 90, 120, 160, 200, 250, Lmax]
                 if 15 <= L <= Lmax})

    pairs = list(itertools.permutations(lakes, 2))
    jobs = [(preproc, window, resp, drv, Es[resp], n,
             D[resp].values.copy(), D[drv].values.copy(), Ls,
             SEED + 991 * i) for i, (resp, drv) in enumerate(pairs)]
    results = _map_jobs(_edge_worker, jobs)

    edges = [r[0] for r in results]
    curves = [c for r in results for c in r[1]]

    ed = pd.DataFrame(edges)
    # 42 simultaneous tests per variant, control the false discovery rate
    ed["q_bh"] = np.round(bh_fdr(ed["p_surrogate"].values), 4)
    ed["significant"] = ed["q_bh"] < ALPHA
    ed["edge"] = (ed["converged"] & ed["significant"]
                  & (ed["rho_Lmax"] >= RHO_FLOOR))


    ed["delta_null_p95"] = round(float(delta_thresh), 4)
    ed["strict_edge"] = ed["edge"] & (ed["delta_rho"] > delta_thresh)
    return ed, pd.DataFrame(curves), pd.DataFrame(Ecurves), Es


def _edge_worker(job):
    """One directed edge: 'resp xmap drv', testing drv -> resp."""
    (preproc, window, resp, drv, E, n, x, y, Ls, seed) = job
    rng_master = np.random.default_rng(seed)
    curves = []

    if True:
        rhos = []
        for L in Ls:
            m, sd = cross_map_rho(x, y, E, TAU, lib_size=L, n_boot=N_BOOT,
                                  exclusion_radius=EXCL,
                                  rng=np.random.default_rng(SEED + L))
            rhos.append(m)
            curves.append({"preproc": preproc, "window": window,
                           "driver": drv, "response": resp,
                           "lib_size": L, "rho_mean": m, "rho_sd": sd})
        rhos = np.asarray(rhos)
        rho_min, rho_max = rhos[0], rhos[-1]
        delta = rho_max - rho_min

        # monotonicity of rho vs L
        c = d = 0
        for a, b in itertools.combinations(range(len(Ls)), 2):
            s = np.sign(rhos[b] - rhos[a])
            c += s > 0
            d += s < 0
        ktau = (c - d) / max(c + d, 1)

        # Ebisuzaki surrogate test at the terminal library size.
        surr = ebisuzaki_surrogates(y, N_SURR, seed=int(rng_master.integers(1e9)))
        null = np.empty(N_SURR)
        M, t_idx = delay_embed(x, E, TAU)
        pos = np.arange(t_idx.size)
        for i in range(N_SURR):
            o, p = simplex_predict(M, t_idx, surr[i], pos, pos, E,
                                   exclusion_radius=EXCL, tp=0)
            null[i] = (np.corrcoef(o, p)[0, 1]
                       if o.size > 2 and np.std(p) > 1e-12 else np.nan)
        null = null[np.isfinite(null)]
        # full-library observed value, comparable to the surrogate draws
        rho_full, _ = cross_map_rho(x, y, E, TAU, lib_size=None,
                                    exclusion_radius=EXCL)
        p_surr = float((1 + (null >= rho_full).sum()) / (1 + null.size))

        # sensitivity: Theiler window of +/- 12 months
        rho_excl12, _ = cross_map_rho(x, y, E, TAU, lib_size=None,
                                      exclusion_radius=EXCL_SENS)

        converged = bool(delta > 0 and ktau > 0)
        rec = {
            "preproc": preproc, "window": window,
            "driver": drv, "response": resp,
            "E_response": E, "n_obs": n,
            "L_min": Ls[0], "L_max": Ls[-1],
            "rho_Lmin": round(rho_min, 4), "rho_Lmax": round(rho_max, 4),
            "rho_full_lib": round(rho_full, 4),
            "delta_rho": round(delta, 4), "kendall_tau_rho_vs_L": round(ktau, 3),
            "p_surrogate": round(p_surr, 4),
            "rho_theiler12": round(rho_excl12, 4),
            "converged": converged,
            "significant_raw_p": bool(p_surr < ALPHA),
        }
    return rec, curves


# adjacency
def adjacency(edges_df, lakes, col="edge"):
    """A[i, j] = rho for 'i xmap j' = strength of j -> i, zero if not an edge."""
    A = pd.DataFrame(0.0, index=lakes, columns=lakes)
    for _, r in edges_df.iterrows():
        if r[col]:
            A.loc[r["response"], r["driver"]] = max(r["rho_Lmax"], 0.0)
    A.index.name = "response_(influenced)"
    A.columns.name = "driver_(influencer)"
    return A


def adjacency_topk(edges_df, lakes, k=TOP_K):
    """Sparse GNAR-ready adjacency: each node keeps its k strongest incoming
    edges by rho, among those that passed the significance test."""
    
    A = pd.DataFrame(0.0, index=lakes, columns=lakes)
    for lk in lakes:
        sub = edges_df[(edges_df.response == lk) & edges_df["significant"]]
        sub = sub.nlargest(k, "rho_Lmax")
        for _, r in sub.iterrows():
            A.loc[lk, r["driver"]] = max(r["rho_Lmax"], 0.0)
    A.index.name = "response_(influenced)"
    A.columns.name = "driver_(influencer)"
    return A


def bh_fdr(p):

    p = np.asarray(p, dtype=float)
    n = p.size
    order = np.argsort(p)
    q = np.empty(n)
    prev = 1.0
    for rank in range(n - 1, -1, -1):
        i = order[rank]
        prev = min(prev, p[i] * n / (rank + 1))
        q[i] = prev
    return np.minimum(q, 1.0)


# negative control 
_NC_N = 367


def _nc_trial(k):

    rng = np.random.default_rng(50000 + k)
    phi = float(rng.uniform(0.65, 0.90))

    def ar1(s):
        r = np.random.default_rng(s)
        x = np.zeros(_NC_N)
        for t in range(1, _NC_N):
            x[t] = phi * x[t - 1] + r.normal()
        return (x - x.mean()) / x.std()

    x, y = ar1(int(rng.integers(1e9))), ar1(int(rng.integers(1e9)))
    E, _, _ = select_E(x, E_RANGE, TAU, EXCL, tol=E_TOL)
    Ls = [20, 45, 90, 160, 250, _NC_N - (E - 1)]
    rr = [cross_map_rho(x, y, E, TAU, lib_size=L, n_boot=60,
                        exclusion_radius=EXCL,
                        rng=np.random.default_rng(SEED + L))[0] for L in Ls]
    surr = ebisuzaki_surrogates(y, N_SURR, seed=int(rng.integers(1e9)))
    M, t_idx = delay_embed(x, E, TAU)
    pos = np.arange(t_idx.size)
    null = []
    for i in range(N_SURR):
        o, p = simplex_predict(M, t_idx, surr[i], pos, pos, E, EXCL, 0)
        if o.size > 2 and np.std(p) > 1e-12:
            null.append(np.corrcoef(o, p)[0, 1])
    null = np.asarray(null)
    rho_full, _ = cross_map_rho(x, y, E, TAU, lib_size=None,
                                exclusion_radius=EXCL)
    p_s = float((1 + (null >= rho_full).sum()) / (1 + null.size))
    return {"trial": k, "E": E, "phi": round(phi, 2),
            "rho_Lmax": round(rr[-1], 4),
            "delta_rho": round(rr[-1] - rr[0], 4),
            "p_surrogate": round(p_s, 4),
            "edge": bool(rr[-1] - rr[0] > 0 and p_s < ALPHA
                         and rr[-1] >= RHO_FLOOR)}


def negative_control(n, n_trials=N_CONTROL):
    """Estimate the false-positive rate empirically."""
    global _NC_N
    _NC_N = n
    return pd.DataFrame(_map_jobs(_nc_trial, list(range(n_trials))))


# main
def main():
    lines = []

    def log(s=""):
        print(s)
        lines.append(str(s))

    log("=" * 76)
    log("PAIRWISE CCM -- directed dependence network, seven African Great Lakes")
    log("=" * 76)
    W = load_levels()
    lakes = list(W.columns)
    log(f"  window {W.index.min():%Y-%m} .. {W.index.max():%Y-%m}  "
        f"({len(W)} months, {len(lakes)} lakes, 0 NaNs)")
    log(f"  train-only window ends {TRAIN_END[:7]} "
        f"({len(W.loc[:TRAIN_END])} months)")
    log(f"  {len(list(itertools.permutations(lakes, 2)))} directed edges per variant"
        f", 4 variants, seed={SEED}")

    # calibration, it defines the strict threshold
    log("\n" + "=" * 76)
    log("NEGATIVE CONTROL: independent AR(1) pairs, lake-like autocorrelation")
    log("=" * 76)
    log("  Identical pipeline on series with NO causal link. Edges here are")
    log(f"  false positives. {N_CONTROL} trials.")
    nc = negative_control(len(W))
    nc.to_csv(os.path.join(OUTDIR, "CCM_10_negative_control.csv"), index=False)
    fpr = nc.edge.mean()
    se = np.sqrt(fpr * (1 - fpr) / len(nc))
    log(f"  false-positive rate of the edge rule (raw p): "
        f"{fpr:.1%}  (+/- {1.96 * se:.1%}, nominal {ALPHA:.0%})")
    log("  surrogate p-value calibration (should track the nominal level):")
    for a in (0.10, 0.05, 0.01):
        log(f"      P(p < {a:.2f}) = {(nc.p_surrogate < a).mean():.3f}"
            f"   nominal {a:.2f}")
    rho_null95 = nc.rho_Lmax.quantile(.95)
    delta_null95 = float(nc.delta_rho.quantile(.95))
    log(f"  rho under the null:   mean {nc.rho_Lmax.mean():.3f}, "
        f"95th pct {rho_null95:.3f}, max {nc.rho_Lmax.max():.3f}")
    log(f"  delta under the null: mean {nc.delta_rho.mean():.3f}, "
        f"95th pct {delta_null95:.3f}")
    log("  --> STRICT convergence threshold: delta must exceed "
        f"{delta_null95:.3f}")

    all_edges, all_curves, all_E = [], [], []
    adjs, adjs_strict = {}, {}
    for preproc in ("stl", "diff"):
        for window in ("full", "train"):
            tag = f"{preproc}_{window}"
            log(f"\n--- variant {tag} ---")
            D = make_variant(W, preproc, window)
            log(f"    n={len(D)} months")
            e, c, ec, Es = run_variant(D, preproc, window, log, delta_null95)
            all_edges.append(e); all_curves.append(c); all_E.append(ec)
            A = adjacency(e, lakes)
            adjs[tag] = A
            A.to_csv(os.path.join(OUTDIR, f"CCM_03_adjacency_{tag}.csv"))
            As = adjacency(e, lakes, col="strict_edge")
            adjs_strict[tag] = As
            As.to_csv(os.path.join(OUTDIR, f"CCM_11_adjacency_strict_{tag}.csv"))
            Ak = adjacency_topk(e, lakes, TOP_K)
            Ak.to_csv(os.path.join(OUTDIR,
                                   f"CCM_09_adjacency_top{TOP_K}_{tag}.csv"))
            log(f"    edges called: {int(e['edge'].sum())} / {len(e)}"
                f"   (converged {int(e['converged'].sum())}, "
                f"significant {int(e['significant'].sum())})"
                f"   | STRICT: {int(e['strict_edge'].sum())}"
                f"   | top-{TOP_K} sparse: {int((Ak.values > 0).sum())} edges")

    E = pd.concat(all_edges, ignore_index=True)
    C = pd.concat(all_curves, ignore_index=True)
    EC = pd.concat(all_E, ignore_index=True)
    E.to_csv(os.path.join(OUTDIR, "CCM_01_edges.csv"), index=False)
    C.to_csv(os.path.join(OUTDIR, "CCM_02_convergence_curves.csv"), index=False)
    EC.to_csv(os.path.join(OUTDIR, "CCM_05_E_selection.csv"), index=False)

    # variant agreement
    log("\n" + "=" * 76)
    log("EDGE COUNTS BY VARIANT")
    log("=" * 76)
    piv = E.pivot_table(index="preproc", columns="window", values="edge",
                        aggfunc="sum")
    log(piv.to_string())

    key = ["driver", "response"]
    wide = E.assign(tag=E.preproc + "_" + E.window).pivot_table(
        index=key, columns="tag", values="edge", aggfunc="first")
    wide["n_variants_calling_edge"] = wide.sum(axis=1)
    wide = wide.reset_index()
    wide.to_csv(os.path.join(OUTDIR, "CCM_06_variant_agreement.csv"), index=False)
    log("\nEdges by number of variants calling them:")
    log(wide["n_variants_calling_edge"].value_counts().sort_index().to_string())
    log("\n  strict-convergence edges by variant:")
    log(E.pivot_table(index="preproc", columns="window", values="strict_edge",
                      aggfunc="sum").to_string())

    # The robust core, edges every variant agrees on
    log("\n" + "=" * 76)
    log("ROBUST CORE -- edges called under ALL FOUR variants")
    log("=" * 76)
    log("  This is the estimated network. Edges called by a single")
    log("  variant are preprocessing artefacts, not findings.")
    core = wide[wide.n_variants_calling_edge == 4][["driver", "response"]].copy()
    st = E[(E.preproc == "stl") & (E.window == "train")].set_index(
        ["driver", "response"])
    core["rho_stl_train"] = [st.loc[(d, r), "rho_Lmax"]
                             for d, r in zip(core.driver, core.response)]
    core["strict_stl_train"] = [bool(st.loc[(d, r), "strict_edge"])
                                for d, r in zip(core.driver, core.response)]
    core = core.sort_values("rho_stl_train", ascending=False)
    core.to_csv(os.path.join(OUTDIR, "CCM_12_robust_core.csv"), index=False)
    log(f"  {len(core)} robust edges:")
    log("  " + ", ".join(f"{SHORT[d]}->{SHORT[r]}({v:.2f})" for d, r, v in
                         zip(core.driver, core.response, core.rho_stl_train)))
    deg = pd.DataFrame({"out_degree": core.groupby("driver").size(),
                        "in_degree": core.groupby("response").size()})
    deg = deg.reindex(lakes).fillna(0).astype(int)
    deg["net_out_minus_in"] = deg.out_degree - deg.in_degree
    deg.to_csv(os.path.join(OUTDIR, "CCM_13_degree.csv"))
    log("\n  in/out degree of the robust core:")
    log(deg.sort_values("net_out_minus_in", ascending=False).to_string())
    fragile = wide[wide.n_variants_calling_edge == 1]
    log(f"\n  fragile edges (one variant only, do NOT report): {len(fragile)}")
    for _, r in fragile.iterrows():
        log(f"    {SHORT[r.driver]} -> {SHORT[r.response]}")

    # Theiler sensitivity
    log("\n" + "=" * 76)
    log("THEILER SENSITIVITY")
    log("=" * 76)
    log("  Excluding library neighbours within +/-12 months of the prediction")
    log("  point removes any skill that is really shared low-frequency or")
    log("  seasonal structure rather than attractor geometry.")
    for tag in ["stl_full", "stl_train", "diff_full", "diff_train"]:
        s = E[(E.preproc + "_" + E.window) == tag]
        drop = s.rho_full_lib - s.rho_theiler12
        log(f"    {tag:11s} mean rho {s.rho_full_lib.mean():.3f} -> "
            f"{s.rho_theiler12.mean():.3f}  (mean drop {drop.mean():+.3f}; "
            f"{int((drop > 0.10).sum())}/{len(s)} edges lose >0.10)")

    # Comparison against the other network estimators
    log("\n" + "=" * 76)
    log("ESTIMATOR AGREEMENT: CCM vs residual correlation vs prewhitened lead-lag")
    log("=" * 76)
    comp = None
    if os.path.exists(EDA_CORR):
        corr = pd.read_csv(EDA_CORR, index_col=0)
        lead = (pd.read_csv(EDA_LEAD) if os.path.exists(EDA_LEAD) else None)
        rows = []
        base = E[(E.preproc == "stl") & (E.window == "full")]
        for a, b in itertools.combinations(lakes, 2):
            ab = base[(base.driver == a) & (base.response == b)].iloc[0]  # a->b
            ba = base[(base.driver == b) & (base.response == a)].iloc[0]  # b->a
            row = {"Lake_x": a, "Lake_y": b,
                   "corr_residual": round(float(corr.loc[a, b]), 3),
                   "ccm_x_to_y_rho": ab["rho_Lmax"], "ccm_x_to_y_edge": ab["edge"],
                   "ccm_y_to_x_rho": ba["rho_Lmax"], "ccm_y_to_x_edge": ba["edge"]}
            row["ccm_any_edge"] = bool(ab["edge"] or ba["edge"])
            row["ccm_dominant_direction"] = (
                f"{SHORT[a]}->{SHORT[b]}" if ab["rho_Lmax"] > ba["rho_Lmax"]
                else f"{SHORT[b]}->{SHORT[a]}")
            if lead is not None:
                m = lead[(lead.Lake_x == a) & (lead.Lake_y == b)]
                if len(m):
                    row["leadlag_peak_lag_m"] = int(m.peak_lag_m.iloc[0])
                    row["leadlag_peak_ccf"] = float(m.peak_ccf.iloc[0])
                    row["leadlag_lead"] = m.lead.iloc[0]
            rows.append(row)
        comp = pd.DataFrame(rows)
        comp.to_csv(os.path.join(OUTDIR, "CCM_04_estimator_agreement.csv"),
                    index=False)
        log(comp.to_string(index=False))
        r = np.corrcoef(comp.corr_residual,
                        comp[["ccm_x_to_y_rho", "ccm_y_to_x_rho"]].max(axis=1))[0, 1]
        log(f"\n  Pearson r( residual correlation , max CCM rho over the two "
            f"directions ) = {r:.3f}")
        log(f"  pairs with at least one CCM edge: "
            f"{int(comp.ccm_any_edge.sum())} / {len(comp)}")
    else:
        log(f"  {EDA_CORR} not found -- comparison skipped")

    log("\n  The null rho 95th percentile "
        f"({rho_null95:.3f}) is the comparison for reading the")
    log("  weaker called edges, an edge with rho near it is weak evidence.")

    # Figures
    _fig_convergence(C, os.path.join(OUTDIR, "CCM_07_convergence_curves.png"))
    _fig_networks(adjs, lakes, os.path.join(OUTDIR, "CCM_08_network_maps.png"))

    with open(os.path.join(OUTDIR, "CCM_00_log.txt"), "w") as f:
        f.write("\n".join(lines))
    log(f"\nDone. Outputs written to: {OUTDIR}")


def _fig_convergence(C, path):
    tags = [("stl", "full"), ("stl", "train"), ("diff", "full"), ("diff", "train")]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=False)
    for ax, (p, w) in zip(axes.ravel(), tags):
        sub = C[(C.preproc == p) & (C.window == w)]
        for (d, r), g in sub.groupby(["driver", "response"]):
            g = g.sort_values("lib_size")
            ax.plot(g.lib_size, g.rho_mean, lw=0.8, alpha=0.55,
                    color="#1f77b4")
        med = sub.groupby("lib_size").rho_mean.median()
        ax.plot(med.index, med.values, lw=2.6, color="crimson", label="median")
        ax.axhline(0, color="grey", lw=0.7)
        ax.set_title(f"{p} / {w}")
        ax.set_xlabel("library size L"); ax.set_ylabel(r"$\rho$  (cross-map skill)")
        ax.legend(fontsize=8)
    fig.suptitle("CCM convergence: cross-map skill against library size\n"
                 "(each thin line is one of the 42 directed edges)", fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def _fig_networks(adjs, lakes, path):
    short = [SHORT[l] for l in lakes]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5.4))
    for ax, (tag, A) in zip(axes, adjs.items()):
        im = ax.imshow(A.values, cmap="viridis", vmin=0,
                       vmax=max(0.35, np.nanmax([a.values.max() for a in adjs.values()])))
        ax.set_xticks(range(len(lakes))); ax.set_xticklabels(short, rotation=45)
        ax.set_yticks(range(len(lakes))); ax.set_yticklabels(short)
        ax.set_title(f"{tag}\n({int((A.values > 0).sum())} edges)")
        ax.set_xlabel("driver  (influencer)")
        ax.set_ylabel("response  (influenced)")
        for i in range(len(lakes)):
            for j in range(len(lakes)):
                v = A.values[i, j]
                if v > 0:
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=7,
                            color="white" if v < 0.25 else "black")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(r"CCM directed adjacency  A[i,j] = $\rho$ for 'i xmap j' "
                 r"= strength of  j $\rightarrow$ i", fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


if __name__ == "__main__":
    main()

PAIRWISE CCM -- directed dependence network, seven African Great Lakes
  window 1995-06 .. 2025-12  (367 months, 7 lakes, 0 NaNs)
  train-only window ends 2020-12 (307 months)
  42 directed edges per variant, 4 variants, seed=20260807

NEGATIVE CONTROL: independent AR(1) pairs, lake-like autocorrelation
  Identical pipeline on series with NO causal link. Edges here are
  false positives. 200 trials.
  false-positive rate of the edge rule (raw p): 4.5%  (+/- 2.9%, nominal 5%)
  surrogate p-value calibration (should track the nominal level):
      P(p < 0.10) = 0.100   nominal 0.10
      P(p < 0.05) = 0.045   nominal 0.05
      P(p < 0.01) = 0.005   nominal 0.01
  rho under the null:   mean 0.055, 95th pct 0.253, max 0.320
  delta under the null: mean 0.021, 95th pct 0.177
  --> STRICT convergence threshold: delta must exceed 0.177

--- variant stl_full ---
    n=367 months
    E(Alb)=5 rho=0.768   [argmax would be E=5, rho=0.768]
    E(Edw)=9 rho=0.649   [argmax would be E=9, rho=0.649]